In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class SimplePointNetEncoder(nn.Module):
    def __init__(self, channel=3):
        super().__init__()
        
        self.conv1 = nn.Conv1d(channel, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 1024, 1)
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(1024)
        
    def forward(self, x):
        # x: (B, channel, N)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.bn3(self.conv3(x))
        x = torch.max(x, 2)[0] # max accross the point dimension(N)
        return x
    
class PointNetCls(nn.Module):
    def __init__(self, k=40, channel=3):
        super().__init__()
        self.encoder = SimplePointNetEncoder(channel)
        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512,256)
        self.fc3 = nn.Linear(256,k)
        self.bn1 = nn.BatchNorm1d(512)
        self.bn2 = nn.BatchNorm1d(256)
        self.dropout = nn.Dropout(p=0.4)
        
    def forward(self, x):
        x = self.encoder(x)
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.dropout(self.fc2(x))))
        x = self.fc3(x)
        return F.log_softmax(x, dim=1)
    
class PointNetLoss(nn.Module):
    def forward(self, pred, target):
        return F.nll_loss(pred, target)
        

In [ ]:
# training and inference